# Picode Training on Google Colab

Train steganography models using free/Pro Colab GPUs with Google Drive for persistence.

**Before starting:**
1. Set runtime to GPU: Runtime > Change runtime type > T4 GPU
2. Have training images ready (COCO train2017 recommended)

Checkpoints save to Google Drive so training survives session disconnects.

## 1. Setup

Mount Google Drive, clone the repo, and install the package.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Clone repo (edit URL to match your fork/repo)
!git clone https://github.com/<user>/picode.git /content/picode 2>/dev/null || \
    (cd /content/picode && git pull)

%cd /content/picode/picode-model
!pip install -q -e '.[lpips,kornia]'

## 2. GPU Check

Verify a GPU is available and check memory.

In [ ]:
!nvidia-smi

import torch
assert torch.cuda.is_available(), "No GPU found! Set runtime to GPU."
print(f"\nGPU: {torch.cuda.get_device_name(0)}")
print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 3. Data Setup

Create the Drive directory structure and get training data.

**Option A** (below): Download COCO train2017 directly in Colab (~18GB, ~20 min)  
**Option B**: Upload images to `My Drive/picode/data/train2017/` via Google Drive  
**Option C**: Skip if data is already on Drive from a previous session

In [ ]:
import os

DRIVE_BASE = "/content/drive/MyDrive/picode"
DATA_DIR = f"{DRIVE_BASE}/data/train2017"
CHECKPOINT_DIR = f"{DRIVE_BASE}/checkpoints"

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Check if data already exists
num_images = len([f for f in os.listdir(DATA_DIR) if f.endswith(('.jpg', '.png'))])
print(f"Found {num_images} images in {DATA_DIR}")

if num_images == 0:
    print("\nNo images found. Run the next cell to download COCO train2017,")
    print("or upload images to My Drive/picode/data/train2017/ manually.")

In [ ]:
# Option A: Download COCO train2017 (~18GB, takes ~20 min)
# Uncomment the lines below to download

# !wget -q http://images.cocodataset.org/zips/train2017.zip -O /content/train2017.zip
# !unzip -q /content/train2017.zip -d /content/drive/MyDrive/picode/data/
# !rm /content/train2017.zip
# print(f"Downloaded {len(os.listdir(DATA_DIR))} images")

## 4. Train (Fresh Start)

Start training from scratch. Checkpoints save to Google Drive every 5000 steps.

In [ ]:
from picode.training.trainer import Trainer

trainer = Trainer.from_config("/content/picode/picode-model/configs/colab_training.yaml")
trainer.fit()

## 5. Resume Training

Resume from the latest checkpoint on Drive (use after a session disconnect).

In [ ]:
import glob
from picode.training.trainer import Trainer

DRIVE_BASE = "/content/drive/MyDrive/picode"
checkpoints = sorted(glob.glob(f"{DRIVE_BASE}/checkpoints/*/checkpoint_*.pt"))

if checkpoints:
    print(f"Resuming from: {checkpoints[-1]}")
    trainer = Trainer.from_checkpoint(checkpoints[-1])
    trainer.fit()
else:
    print("No checkpoints found. Use cell 4 to start fresh.")

## 6. TensorBoard

Monitor training metrics inline.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/drive/MyDrive/picode/checkpoints/runs

## 7. Evaluate

Run evaluation on the best checkpoint.

In [ ]:
import glob
from picode.training.trainer import Trainer

DRIVE_BASE = "/content/drive/MyDrive/picode"
best_checkpoints = sorted(glob.glob(f"{DRIVE_BASE}/checkpoints/*/best.pt"))

if best_checkpoints:
    print(f"Evaluating: {best_checkpoints[-1]}")
    trainer = Trainer.from_checkpoint(best_checkpoints[-1])
    metrics = trainer.evaluate()
    print(f"\nResults:\n  Bit accuracy: {metrics.bit_accuracy:.4f}")
    print(f"  PSNR: {metrics.psnr:.2f} dB")
    print(f"  SSIM: {metrics.ssim:.4f}")
else:
    print("No best.pt found. Train first or check checkpoint directory.")

## 8. Download Checkpoint

Download the best checkpoint to your local machine.

In [ ]:
import glob
from google.colab import files

DRIVE_BASE = "/content/drive/MyDrive/picode"
best_checkpoints = sorted(glob.glob(f"{DRIVE_BASE}/checkpoints/*/best.pt"))

if best_checkpoints:
    print(f"Downloading: {best_checkpoints[-1]}")
    files.download(best_checkpoints[-1])
else:
    print("No best.pt found.")